# 02 — Complete RAG Pipeline: Data Injection → Vector DB → Retrieval → LLM

This notebook builds the **entire RAG pipeline** from the crash course, step by step:

```
                    DATA INJECTION PIPELINE
  PDFs ──▶ Data Parsing ──▶ Chunking ──▶ Embeddings ──▶ Vector Store (ChromaDB)

                    QUERY RETRIEVAL PIPELINE
  User query ──▶ Embedding ──▶ Similarity Search ──▶ Context ──▶ Prompt + LLM ──▶ Answer
                 (same model)     (vector store)      (Retrieval)  (Augmentation)  (Generation)
```

Components built in this notebook (as reusable classes, mirroring the video):

| Component | Role |
|---|---|
| `process_all_pdfs()` | Data ingestion — PDFs → LangChain Documents |
| `split_documents()` | Chunking with `RecursiveCharacterTextSplitter` |
| `EmbeddingManager` | Open-source `all-MiniLM-L6-v2` sentence-transformer (384-dim) |
| `VectorStore` | ChromaDB persistent vector database |
| `RAGRetriever` | Query → embedding → similarity search → context |
| `rag_simple()` / `rag_advanced()` | Augmentation + generation with a Groq LLM |

## 0. Imports

In [1]:
import os
import uuid
from pathlib import Path
from typing import Any, Dict, List

import chromadb
import numpy as np
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully")

All libraries imported successfully


/var/folders/g7/ps83y1v13d30d1_46t_z2z6m0000gp/T/ipykernel_5259/2650881676.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Data Ingestion — read all PDFs in a directory

`process_all_pdfs` walks a folder, loads every PDF with `PyPDFLoader`
(one `Document` **per page**) and enriches each Document's metadata with
`source_file` and `file_type` so every chunk stays traceable after retrieval.

In [2]:
def process_all_pdfs(pdf_directory: str) -> List[Any]:
    """Load every PDF in `pdf_directory` into LangChain Documents.

    Args:
        pdf_directory: Folder to scan (recursively) for .pdf files.

    Returns:
        A flat list of Documents — one per PDF page — each carrying the
        default PyPDF metadata plus our extra `source_file` / `file_type` keys.
    """
    pdf_dir = Path(pdf_directory)
    pdf_files = sorted(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")

    all_documents = []
    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()  # one Document per page

        # Enrich metadata: these keys can later be used as *filters*
        # during similarity search in the vector store.
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"

        print(f"  {pdf_file.name}: {len(documents)} pages")
        all_documents.extend(documents)

    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdf")
print(f"\nTotal documents (pages): {len(all_pdf_documents)}")

Found 3 PDF files
  attention_is_all_you_need.pdf: 3 pages
  embedding_models_report.pdf: 3 pages
  rag_systems_overview.pdf: 2 pages

Total documents (pages): 8


In [3]:
# Inspect one document: page_content holds the text, metadata holds the trace info.
first = all_pdf_documents[0]
print("METADATA:", first.metadata)
print()
print("PAGE CONTENT (first 300 chars):")
print(first.page_content[:300])

METADATA: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/attention_is_all_you_need.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need.pdf', 'file_type': 'pdf'}

PAGE CONTENT (first 300 chars):
Attention Is All You Need - An Overview
The Transformer is a neural network architecture based entirely on attention mechanisms,
dispensing with recurrence and convolutions entirely. The dominant sequence transduction
models before the Transformer were based on complex recurrent or convolutional neu


## 2. Chunking — `RecursiveCharacterTextSplitter`

Embedding models and LLMs both have a **fixed context size**, so we can never embed a
whole PDF in one shot. Chunking splits the documents into ~1000-character pieces with a
**200-character overlap** so no sentence loses its surrounding context at a boundary.

The *recursive* splitter tries separators **in order** — paragraph break (`\n\n`),
then newline, then space, then any character — so chunks break at the most natural
point available.

In [4]:
def split_documents(documents: List[Any], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Any]:
    """Split Documents into smaller overlapping chunks.

    Args:
        documents: Documents from the ingestion step.
        chunk_size: Max characters per chunk.
        chunk_overlap: Characters shared between consecutive chunks.

    Returns:
        The chunked Documents (each keeps its parent's metadata).
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],  # tried in order, most natural first
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("\nExample chunk:")
        print("  content:", split_docs[0].page_content[:200].replace("\n", " "))
        print("  metadata:", split_docs[0].metadata)
    return split_docs


chunks = split_documents(all_pdf_documents)

Split 8 documents into 8 chunks

Example chunk:
  content: Attention Is All You Need - An Overview The Transformer is a neural network architecture based entirely on attention mechanisms, dispensing with recurrence and convolutions entirely. The dominant sequ
  metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/attention_is_all_you_need.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need.pdf', 'file_type': 'pdf'}


## 3. Embeddings — `EmbeddingManager`

Embeddings convert text into **vectors** (numerical representations) so we can run
similarity search / cosine similarity against them.

We use the fully **open-source** Hugging Face model **`all-MiniLM-L6-v2`** via
`sentence-transformers`: small, fast, and it maps any text to a **384-dimensional**
vector. No API key, no cost — it runs locally.

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using a sentence-transformer model."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.

        Args:
            model_name: Hugging Face model name for sentence embeddings.
                        all-MiniLM-L6-v2 produces 384-dimensional vectors.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()  # load immediately so the manager is ready to use

    def _load_model(self):
        """Load the sentence-transformer model (protected helper).

        The leading underscore marks this as internal: it is only meant to be
        called from inside the class.
        """
        print(f"Loading embedding model: {self.model_name} ...")
        self.model = SentenceTransformer(self.model_name)
        dim = self.model.get_sentence_embedding_dimension()
        print(f"Model loaded successfully. Embedding dimension: {dim}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args:
            texts: The strings to embed.

        Returns:
            numpy array of shape (len(texts), embedding_dim).
        """
        if self.model is None:
            raise ValueError("Model not loaded")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


/var/folders/g7/ps83y1v13d30d1_46t_z2z6m0000gp/T/ipykernel_5259/2438620515.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


## 4. Vector Store — ChromaDB (persistent)

The vector store saves each chunk's **embedding + text + metadata** and answers
similarity-search queries. We use **ChromaDB** with a `PersistentClient`, meaning the
store is written to disk (`../data/vector_store`) — restart the notebook and the data
is still there, no re-embedding needed.

> The modular `src/` pipeline uses **FAISS** for the same job — the two are
> interchangeable at this layer.

In [6]:
class VectorStore:
    """Manages document embeddings in a persistent ChromaDB collection."""

    def __init__(self, collection_name: str = "pdf_documents",
                 persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection (like a table).
            persist_directory: Folder where ChromaDB persists to disk.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Create the ChromaDB client and collection (protected helper)."""
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        # cosine space => distance = 1 - cosine_similarity, so the
        # `similarity = 1 - distance` conversion in the retriever is exact.
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document chunks for RAG",
                "hnsw:space": "cosine",
            },
        )
        print(f"Collection ready: {self.collection_name}")
        print(f"Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add chunked documents and their embeddings to the vector store.

        Args:
            documents: The chunked LangChain Documents.
            embeddings: One embedding row per document (same order!).
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        # ChromaDB wants four parallel lists: ids, embeddings, metadatas, texts.
        ids, metadatas, texts_list, embeddings_list = [], [], [], []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(str(uuid.uuid4()))          # unique id per record
            metadata = dict(doc.metadata)           # copy the chunk's metadata
            metadata["doc_index"] = i               # extra bookkeeping fields
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            texts_list.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=texts_list,
        )
        print(f"Inserted {len(documents)} documents. "
              f"Total documents in collection: {self.collection.count()}")


vectorstore = VectorStore()

Collection ready: pdf_documents
Existing documents in collection: 0


### Run the injection: chunks → embeddings → vector store

This completes the **data injection pipeline** — after this cell, the knowledge base
lives on disk as vectors.

In [7]:
# Extract the raw text of every chunk...
texts = [doc.page_content for doc in chunks]

# ...convert the texts to embeddings...
embeddings = embedding_manager.generate_embeddings(texts)

# ...and store text + metadata + vectors in the vector database.
# (Re-running this cell re-inserts the chunks under new UUIDs — to rebuild from
#  scratch, delete the ../data/vector_store folder first.)
vectorstore.add_documents(chunks, embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (8, 384)
Inserted 8 documents. Total documents in collection: 8


## 5. Retrieval — `RAGRetriever`

The retriever is a thin interface **on top of** the vector store. For every query it:

1. embeds the query with the **same** embedding model used at injection time,
2. runs similarity search in ChromaDB,
3. converts distance → similarity (`similarity = 1 − distance`),
4. keeps only results above `score_threshold` and returns them as the **context**.

In [8]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Wire the retriever to a vector store and an embedding manager.

        Args:
            vector_store: The populated VectorStore to search in.
            embedding_manager: Must be the same model used at injection time,
                otherwise query and document vectors are incomparable.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5,
                 score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve the most relevant document chunks for a query.

        Args:
            query: The search query text.
            top_k: Maximum number of chunks to return.
            score_threshold: Minimum similarity score (0..1) to keep a hit.

        Returns:
            A list of dicts with id, content, metadata, similarity_score,
            distance and rank — the "context" handed to the LLM later.
        """
        # 1. Convert the query into a vector (same embedding space as the docs)
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # 2. Similarity search in ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
        )

        # 3. Unpack + convert cosine distance to similarity score
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            zipped = zip(
                results["ids"][0],
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
            for doc_id, document, metadata, distance in zipped:
                similarity_score = 1 - distance  # cosine: distance = 1 - similarity
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": len(retrieved_docs) + 1,
                    })
        print(f"Retrieved {len(retrieved_docs)} documents "
              f"(top_k={top_k}, threshold={score_threshold})")
        return retrieved_docs


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [9]:
# Test retrieval — this query should hit the attention/transformer PDF.
results = rag_retriever.retrieve("What is the attention mechanism?", top_k=3)

for doc in results:
    print(f"[rank {doc['rank']}] score={doc['similarity_score']:.4f} "
          f"source={doc['metadata'].get('source_file')} page={doc['metadata'].get('page')}")
    print("   ", doc["content"][:200].replace("\n", " "), "...")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.0)
[rank 1] score=0.5374 source=attention_is_all_you_need.pdf page=2
    Why Self-Attention Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. S ...

[rank 2] score=0.4690 source=attention_is_all_you_need.pdf page=0
    Attention Is All You Need - An Overview The Transformer is a neural network architecture based entirely on attention mechanisms, dispensing with recurrence and convolutions entirely. The dominant sequ ...

[rank 3] score=0.4086 source=attention_is_all_you_need.pdf page=1
    Scaled Dot-Product Attention The particular attention used in the Transformer is called Scaled Dot-Product Attention. The input consists of queries and keys of dimension dk, and values of dimension dv ...



In [10]:
# Another query — should hit the embedding-models report.
results = rag_retriever.retrieve("unified multitask learning framework", top_k=3)

for doc in results:
    print(f"[rank {doc['rank']}] score={doc['similarity_score']:.4f} "
          f"source={doc['metadata'].get('source_file')} page={doc['metadata'].get('page')}")
    print("   ", doc["content"][:200].replace("\n", " "), "...")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.0)
[rank 1] score=0.4521 source=embedding_models_report.pdf page=0
    Text Embedding Models - Technical Report Text embedding models convert natural language into dense numerical vectors that capture semantic meaning. Similar texts map to nearby points in the vector spa ...

[rank 2] score=0.3785 source=attention_is_all_you_need.pdf page=1
    Scaled Dot-Product Attention The particular attention used in the Transformer is called Scaled Dot-Product Attention. The input consists of queries and keys of dimension dk, and values of dimension dv ...

[rank 3] score=0.3407 source=rag_systems_overview.pdf page=0
    Retrieval Augmented Generation (RAG) - Overview Retrieval Augmented Generation is the process of optimizing the output of a large language model by referencing an authoritative knowledge base outside  ...



## 6. Generation — plug in the LLM (Groq)

We now complete **R-A-G**: the retrieved chunks (context) are combined with a prompt
(**Augmentation**) and sent to an LLM (**Generation**).

Like the crash course, we use **Groq**, which hosts open models behind a fast, free
API. Put your key in `RAG/.env` (`GROQ_API_KEY=...`, free at
https://console.groq.com/keys).

> **On model names:** Groq rotates its catalog — the video's `gemma2-9b-it` and its
> successor `llama-3.1-8b-instant` are both gone. Rather than hard-code a name that
> expires, we call `src.search.resolve_model()`, which asks Groq what this key can
> actually use and picks the best available open model. The cell also degrades
> gracefully: with no key, the LLM steps are skipped and retrieval still works.

In [11]:
import sys

sys.path.append("..")  # make the src/ package importable from notebooks/

from dotenv import load_dotenv
from src.search import resolve_model

load_dotenv("../.env")  # loads GROQ_API_KEY into the environment if present

groq_api_key = os.getenv("GROQ_API_KEY")
llm = None
if groq_api_key and groq_api_key != "your_groq_api_key_here":
    from langchain_groq import ChatGroq

    model_name = resolve_model(groq_api_key)  # what can this key actually use?
    llm = ChatGroq(
        api_key=groq_api_key,
        model=model_name,
        temperature=0.1,      # low temp = stay grounded in the context
        max_tokens=1024,      # generous: gpt-oss models reason before answering
    )
    print("Groq LLM initialized:", model_name)
else:
    print("GROQ_API_KEY not set — LLM cells below will be skipped (retrieval still works)")

Groq LLM initialized: openai/gpt-oss-120b


### 6a. `rag_simple` — the minimal RAG function

Retrieve → join the chunks into one context block → format the prompt → invoke the LLM.

In [12]:
def rag_simple(query: str, retriever: RAGRetriever, llm, top_k: int = 3) -> str:
    """Minimal RAG: retrieve context and generate a grounded answer.

    Args:
        query: The user's question.
        retriever: A RAGRetriever wired to the populated vector store.
        llm: A LangChain chat model (ChatGroq here).
        top_k: How many chunks to use as context.

    Returns:
        The LLM's answer string.
    """
    # RETRIEVAL: get the most relevant chunks
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # AUGMENTATION: context + instruction prompt
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    # GENERATION: the LLM writes the final answer
    response = llm.invoke(prompt)
    return response.content


if llm is not None:
    answer = rag_simple("What is the attention mechanism?", rag_retriever, llm)
    print("\nANSWER:\n", answer)
else:
    print("Skipped (no GROQ_API_KEY)")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.0)



ANSWER:
 The attention mechanism maps a **query** together with a set of **key‑value pairs** to an output vector. It computes a compatibility score between the query and each key, normalizes these scores (e.g., with softmax) to obtain attention weights, and then returns a weighted sum of the corresponding values. This lets the model focus on the most relevant parts of the input when producing each representation.


### 6b. `rag_advanced` — answers with sources and confidence

Same flow, but the response now includes **where the answer came from** (source file,
page, similarity score, preview) and a **confidence score** (the best similarity among
the retrieved chunks) — the pieces you need to make a RAG system trustworthy.

In [13]:
def rag_advanced(query: str, retriever: RAGRetriever, llm, top_k: int = 3,
                 min_score: float = 0.1, return_context: bool = False) -> Dict[str, Any]:
    """Enhanced RAG pipeline returning answer + sources + confidence.

    Args:
        query: The user's question.
        retriever: A RAGRetriever wired to the populated vector store.
        llm: A LangChain chat model.
        top_k: How many chunks to retrieve.
        min_score: Minimum similarity for a chunk to be used.
        return_context: If True, also return the full context string.

    Returns:
        dict with keys: answer, sources, confidence, (optionally) context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {"answer": "No relevant context found.", "sources": [],
                "confidence": 0.0, "context": ""}

    context = "\n\n".join(doc["content"] for doc in results)
    sources = [
        {
            "source_file": doc["metadata"].get("source_file", "unknown"),
            "page": doc["metadata"].get("page", "n/a"),
            "score": round(doc["similarity_score"], 4),
            "preview": doc["content"][:300],
        }
        for doc in results
    ]
    confidence = max(doc["similarity_score"] for doc in results)

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    output = {"answer": response.content, "sources": sources,
              "confidence": round(confidence, 4)}
    if return_context:
        output["context"] = context
    return output


if llm is not None:
    result = rag_advanced("What is the attention mechanism?", rag_retriever, llm,
                          return_context=True)
    print("ANSWER:\n", result["answer"])
    print("\nCONFIDENCE:", result["confidence"])
    print("\nSOURCES:")
    for s in result["sources"]:
        print(f"  - {s['source_file']} (page {s['page']}, score {s['score']})")
else:
    print("Skipped (no GROQ_API_KEY)")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.1)


ANSWER:
 The attention mechanism maps a **query** together with a set of **key‑value pairs** to an output vector. It computes a compatibility score between the query and each key, normalizes these scores (e.g., with softmax) to obtain attention weights, and then returns a weighted sum of the corresponding values. This lets the model focus on the most relevant parts of the input when producing each representation.

CONFIDENCE: 0.5374

SOURCES:
  - attention_is_all_you_need.pdf (page 2, score 0.5374)
  - attention_is_all_you_need.pdf (page 0, score 0.469)
  - attention_is_all_you_need.pdf (page 1, score 0.4086)


In [14]:
# Try a query that targets a different document — the embeddings report.
if llm is not None:
    result = rag_advanced("hard negative mining techniques", rag_retriever, llm,
                          min_score=0.1)
    print("ANSWER:\n", result["answer"])
    print("\nSOURCES:")
    for s in result["sources"]:
        print(f"  - {s['source_file']} (page {s['page']}, score {s['score']})")
else:
    print("Skipped (no GROQ_API_KEY)")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.1)


ANSWER:
 **Hard negative mining techniques**

- **Source from a first‑stage BM25 retriever** – pull passages that are lexically similar to the query but not relevant.  
- **Use a pre‑trained dense retriever** – select high‑scoring passages that the model mistakenly ranks as relevant.  
- **Refresh the negative pool periodically** – as the model improves, regenerate hard negatives so they stay challenging.  

These strategies provide “hard” negatives—texts that look like answers but aren’t—forcing the embedding model to learn finer semantic distinctions and boosting retrieval performance.

SOURCES:
  - embedding_models_report.pdf (page 1, score 0.7348)
  - embedding_models_report.pdf (page 0, score 0.1188)
  - rag_systems_overview.pdf (page 1, score 0.1141)


### 6c. `rag_pipeline_advanced` — streaming, citations, history, summarization

The last pipeline from the video adds the "product features" on top:

* **streaming** — print tokens as the LLM produces them,
* **citations** — inline `[n]` markers mapped to the retrieved sources,
* **history** — keep a conversation log across calls,
* **summarization** — optionally compress the answer to one sentence.

In [15]:
conversation_history: List[Dict[str, str]] = []  # persists across calls


def rag_pipeline_advanced(query: str, retriever: RAGRetriever, llm, top_k: int = 3,
                          min_score: float = 0.1, stream: bool = False,
                          summarize: bool = False) -> Dict[str, Any]:
    """Full-featured RAG pipeline: citations + history + optional streaming/summary.

    Args:
        query: The user's question.
        retriever: A RAGRetriever wired to the populated vector store.
        llm: A LangChain chat model.
        top_k: How many chunks to retrieve.
        min_score: Minimum similarity for a chunk to count as context.
        stream: If True, print the answer token-by-token as it is generated.
        summarize: If True, ask the LLM for a one-sentence summary of its answer.

    Returns:
        dict with keys: answer, citations, summary, history_length.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {"answer": "No relevant context found.", "citations": [],
                "summary": None, "history_length": len(conversation_history)}

    # Number each chunk so the LLM can cite them as [1], [2], ...
    context = "\n\n".join(
        f"[{i + 1}] {doc['content']}" for i, doc in enumerate(results)
    )
    citations = [
        {"id": i + 1,
         "source_file": doc["metadata"].get("source_file", "unknown"),
         "page": doc["metadata"].get("page", "n/a"),
         "score": round(doc["similarity_score"], 4)}
        for i, doc in enumerate(results)
    ]

    # Include recent conversation history for follow-up questions.
    history_text = "\n".join(
        f"User: {turn['query']}\nAssistant: {turn['answer']}"
        for turn in conversation_history[-3:]
    )

    prompt = f"""Use the numbered context passages to answer the question concisely.
Cite the passages you used with their number, e.g. [1].

{f"Previous conversation:{chr(10)}{history_text}{chr(10)}" if history_text else ""}Context:
{context}

Question: {query}

Answer:"""

    if stream:
        # GENERATION (streaming): print chunks as they arrive
        pieces = []
        for token in llm.stream(prompt):
            print(token.content, end="", flush=True)
            pieces.append(token.content)
        print()
        answer = "".join(pieces)
    else:
        answer = llm.invoke(prompt).content

    summary = None
    if summarize:
        summary = llm.invoke(
            f"Summarize the following answer in one sentence:\n\n{answer}"
        ).content

    conversation_history.append({"query": query, "answer": answer})
    return {"answer": answer, "citations": citations, "summary": summary,
            "history_length": len(conversation_history)}


if llm is not None:
    out = rag_pipeline_advanced("What is attention is all you need?",
                                rag_retriever, llm, stream=True, summarize=True)
    print("\nCITATIONS:")
    for c in out["citations"]:
        print(f"  [{c['id']}] {c['source_file']} (page {c['page']}, score {c['score']})")
    print("\nSUMMARY:", out["summary"])
    print("History length:", out["history_length"])
else:
    print("Skipped (no GROQ_API_KEY)")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (top_k=3, threshold=0.1)


“

Attention

Is

...

 vectors

[

1

].


CITATIONS:
  [1] attention_is_all_you_need.pdf (page 0, score 0.415)
  [2] attention_is_all_you_need.pdf (page 2, score 0.4137)
  [3] attention_is_all_you_need.pdf (page 1, score 0.3757)

SUMMARY: “Attention Is All You Need” introduces the Transformer—a neural‑network model that eliminates recurrent and convolutional layers by using attention mechanisms to compute each output as a weighted sum of value vectors, with weights determined by the compatibility between query and key vectors.
History length: 1


## 7. Wrap-up

You just built the complete traditional RAG system:

1. **Data injection pipeline** — `process_all_pdfs` → `split_documents` →
   `EmbeddingManager` → `VectorStore` (persistent ChromaDB).
2. **Query retrieval pipeline** — `RAGRetriever` → context → prompt → Groq LLM →
   answer (+ sources, confidence, citations, history, streaming, summarization).

**Assignment (from the crash course):** rebuild this pipeline for another data format —
CSV, Excel, JSON, or a SQL database — by only swapping the loader in step 1. Everything
downstream stays identical.

**Next step:** the same pipeline as importable modules — see `src/` (`data_loader.py`,
`embedding.py`, `vector_store.py` with FAISS, `search.py`) and run `python app.py`.